#DataSet Preparation

Import Library

In [ ]:
import pandas as pd
import re
import nltk

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Load Both CSV Files

In [ ]:
eng1 = pd.read_csv('/content/English Phishing Email Dataset1.csv')
eng2 = pd.read_csv('/content/English Phishing Email Dataset2.csv')
malay1 = pd.read_csv('/content/Malay Phishing Email Dataset1.csv')
malay2 = pd.read_csv('/content/Malay Phishing Email Dataset2.csv')

In [ ]:
print("English 1:", eng1.shape)
print("English 2:", eng2.shape)
print("Malay 1:", malay1.shape)
print("Malay 2:", malay2.shape)

English 1: (5730, 110)
English 2: (5171, 4)
Malay 1: (1240, 2)
Malay 2: (6200, 2)


In [ ]:
print("English 1 columns:", eng1.columns)
print("English 2 columns:", eng2.columns)
print("Malay 1 columns:", malay1.columns)
print("Malay 2 columns:", malay2.columns)

English 1 columns: Index(['text', 'label', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5',
       'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9',
       ...
       'Unnamed: 100', 'Unnamed: 101', 'Unnamed: 102', 'Unnamed: 103',
       'Unnamed: 104', 'Unnamed: 105', 'Unnamed: 106', 'Unnamed: 107',
       'Unnamed: 108', 'Unnamed: 109'],
      dtype='object', length=110)
English 2 columns: Index(['Unnamed: 0', 'label', 'text', 'label_num'], dtype='object')
Malay 1 columns: Index(['text', 'label'], dtype='object')
Malay 2 columns: Index(['text', 'label'], dtype='object')


Keep Only Needed Columns

In [ ]:
eng1 = eng1[['text', 'label']]
eng2 = eng2[['text', 'label']]
malay1 = malay1[['text', 'label']]
malay2 = malay2[['text', 'label']]

In [ ]:
print(eng1.columns)
print(malay1.columns)
print(eng2.columns)
print(malay2.columns)

Index(['text', 'label'], dtype='object')
Index(['text', 'label'], dtype='object')
Index(['text', 'label'], dtype='object')
Index(['text', 'label'], dtype='object')


Clean English Labels

In [ ]:
eng1['label'] = eng1['label'].astype(str).str.strip()

eng1 = eng1[eng1['label'].isin(['0', '1'])]

eng1['label'] = eng1['label'].map({
    '0': 'safe',
    '1': 'malicious'
})

eng1['language'] = 'English'
eng1['source'] = 'English Dataset 1'

In [ ]:
print(eng2['label'].value_counts(dropna=False))
print(eng2['label'].unique())
eng2['label'] = eng2['label'].astype(str).str.lower().str.strip()

eng2['label'] = eng2['label'].map({
    '0': 'safe',
    '1': 'malicious',
    'ham': 'safe',
    'spam': 'malicious',
    'safe': 'safe',
    'malicious': 'malicious'
})

eng2['language'] = 'English'
eng2['source'] = 'English Dataset 2'

label
safe         3672
malicious    1499
Name: count, dtype: int64
['safe' 'malicious']


Clean Malay Labels

In [ ]:
malay1['label'] = malay1['label'].astype(str).str.lower().str.strip()

malay1['label'] = malay1['label'].map({
    'legit': 'safe',
    'phishing': 'malicious',
    'safe': 'safe',
    'malicious': 'malicious',
    '0': 'safe',
    '1': 'malicious'
})

malay1['language'] = 'Malay'
malay1['source'] = 'Malay Dataset 1'

In [ ]:
malay2['label'] = malay2['label'].astype(str).str.lower().str.strip()

malay2['label'] = malay2['label'].map({
    'legit': 'safe',
    'phishing': 'malicious',
    'safe': 'safe',
    'malicious': 'malicious',
    '0': 'safe',
    '1': 'malicious'
})

malay2['language'] = 'Malay'
malay2['source'] = 'Malay Dataset 2'

Check Each Dataset After Label Cleaning

In [ ]:
datasets = {
    "English Dataset 1": eng1,
    "English Dataset 2": eng2,
    "Malay Dataset 1": malay1,
    "Malay Dataset 2": malay2
}

for name, df in datasets.items():
    print("\n", name)
    print(df.shape)
    print(df['label'].value_counts(dropna=False))


 English Dataset 1
(5726, 4)
label
safe         4358
malicious    1368
Name: count, dtype: int64

 English Dataset 2
(5171, 4)
label
safe         3672
malicious    1499
Name: count, dtype: int64

 Malay Dataset 1
(1240, 4)
label
safe         620
malicious    620
Name: count, dtype: int64

 Malay Dataset 2
(6200, 4)
label
malicious    3226
safe         2974
Name: count, dtype: int64


Combine English + Malay

In [ ]:
combined_df = pd.concat([eng1, eng2, malay1, malay2], ignore_index=True)

print(combined_df.shape)
print(combined_df.head())

(18337, 4)
                                                text      label language  \
0  Subject: naturally irresistible your corporate...  malicious  English   
1  Subject: the stock trading gunslinger  fanny i...  malicious  English   
2  Subject: unbelievable new homes made easy  im ...  malicious  English   
3  Subject: 4 color printing special  request add...  malicious  English   
4  Subject: do not have money , get software cds ...  malicious  English   

              source  
0  English Dataset 1  
1  English Dataset 1  
2  English Dataset 1  
3  English Dataset 1  
4  English Dataset 1  


Remove Missing Value + Duplicate

In [ ]:
combined_df = combined_df.dropna(subset=['text', 'label'])

combined_df = combined_df.drop_duplicates(subset=['text'])

combined_df = combined_df.reset_index(drop=True)

print("After cleaning:")
print(combined_df.shape)
print(combined_df['label'].value_counts())
print(combined_df['language'].value_counts())

After cleaning:
(12389, 4)
label
safe         8621
malicious    3768
Name: count, dtype: int64
language
English    10686
Malay       1703
Name: count, dtype: int64


Check Class and Language Distribution

In [ ]:
print("Class distribution:")
print(combined_df['label'].value_counts())

print("\nLanguage distribution:")
print(combined_df['language'].value_counts())

print("\nLanguage + label distribution:")
print(pd.crosstab(combined_df['language'], combined_df['label']))

Class distribution:
label
safe         8621
malicious    3768
Name: count, dtype: int64

Language distribution:
language
English    10686
Malay       1703
Name: count, dtype: int64

Language + label distribution:
label     malicious  safe
language                 
English        2830  7856
Malay           938   765


Balance Dataset to 55/45

In [ ]:
malicious_df = combined_df[combined_df['label'] == 'malicious']

malay_safe_df = combined_df[
    (combined_df['label'] == 'safe') &
    (combined_df['language'] == 'Malay')
]

english_safe_df = combined_df[
    (combined_df['label'] == 'safe') &
    (combined_df['language'] == 'English')
]

malicious_count = len(malicious_df)

# For 55% safe and 45% malicious:
target_safe_count = int((55 / 45) * malicious_count)

english_safe_needed = target_safe_count - len(malay_safe_df)

english_safe_sampled = english_safe_df.sample(
    n=english_safe_needed,
    random_state=42
)

balanced_df = pd.concat(
    [malicious_df, malay_safe_df, english_safe_sampled],
    ignore_index=True
)

balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanced dataset:")
print(balanced_df.shape)
print(balanced_df['label'].value_counts())
print(balanced_df['language'].value_counts())
print(pd.crosstab(balanced_df['language'], balanced_df['label']))

Balanced dataset:
(8373, 4)
label
safe         4605
malicious    3768
Name: count, dtype: int64
language
English    6670
Malay      1703
Name: count, dtype: int64
label     malicious  safe
language                 
English        2830  3840
Malay           938   765


Create Stopwords

In [ ]:
english_stopwords = set(stopwords.words('english'))

malay_stopwords = {
    'yang', 'dan', 'di', 'ke', 'dari', 'untuk', 'dengan', 'ini', 'itu',
    'anda', 'saya', 'kami', 'kita', 'adalah', 'akan', 'telah', 'sebagai',
    'dalam', 'pada', 'tidak', 'boleh', 'atau', 'juga', 'lebih', 'serta',
    'kerana', 'oleh', 'hingga', 'sila', 'adakah', 'kepada', 'mereka',
    'semua', 'sudah', 'belum', 'lagi', 'jika', 'seperti', 'hanya',
    'tersebut', 'ialah', 'iaitu'
}

all_stopwords = english_stopwords.union(malay_stopwords)

Text Cleaning Function

In [ ]:
def clean_text(text):
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove numbers
    text = re.sub(r'\d+', ' ', text)

    # Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords and very short words
    tokens = [
        word for word in tokens
        if word not in all_stopwords and len(word) > 2
    ]

    return ' '.join(tokens)

Apply Text Cleaning

In [ ]:
balanced_df['clean_text'] = balanced_df['text'].apply(clean_text)

balanced_df[['text', 'clean_text', 'label', 'language', 'source']].head()

,text,clean_text,label,language,source
0,Subject: enron / hpl actuals for september 26 ...,subject enron hpl actuals september tuesday se...,safe,English,English Dataset 2
1,Subject: 17 mens and 81 womans die from aiw .\...,subject mens womans die aiw tre lnk paliourg t...,malicious,English,English Dataset 2
2,Subject: re : greetings from london ( to enron...,subject greetings london enron iris please fee...,safe,English,English Dataset 1
3,"Subject: digital cameras , pda ' s and mp 3 pl...",subject digital cameras pda player drive type ...,malicious,English,English Dataset 2
4,Subject: interview - numerical methods & finan...,subject interview numerical methods finance de...,safe,English,English Dataset 1


Remove Empty Cleaned Text

In [ ]:
balanced_df = balanced_df[balanced_df['clean_text'].str.strip() != '']

balanced_df = balanced_df.reset_index(drop=True)

print("Final cleaned dataset:")
print(balanced_df.shape)
print(balanced_df['label'].value_counts())
print(balanced_df['language'].value_counts())
print(pd.crosstab(balanced_df['language'], balanced_df['label']))

Final cleaned dataset:
(8373, 5)
label
safe         4605
malicious    3768
Name: count, dtype: int64
language
English    6670
Malay      1703
Name: count, dtype: int64
label     malicious  safe
language                 
English        2830  3840
Malay           938   765


Final Save

In [ ]:
balanced_df.to_csv('final_balanced_multilingual_email_dataset.csv', index=False)

In [ ]:
from google.colab import files

files.download('final_balanced_multilingual_email_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>